# 16.10 怎麼用重算換記憶體？

# 第 16 章：由量測帶出效率 BKM

前置：已有第4章能forward與生成的模型。像整理廚房，先找哪一步最慢再改工作流程。CPU小例只驗證數值與資料契約；GPU速度、峯值記憶體與編譯收益必須在讀者的硬體實測。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：舊筆記不用重寫：只加新字的K/V**

cache保存每層筆記；不是提前知道下一字答案。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/cache.svg")))

**先想一想：**少存中間結果，會免費節省而不增加運算嗎？

訓練需保存forward中間結果供backward。checkpointing只留部分，backward再算一次，像少存筆記、多重做一遍。不是模型檔案checkpoint。

**把直覺寫成數學：**時間換activation memory；dropout等隨機狀態要保持可重現。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from torch.utils.checkpoint import checkpoint

layer = nn.Sequential(nn.Linear(4, 8), nn.GELU(), nn.Linear(8, 4))
x = torch.randn(2, 4, requires_grad=True)
y = checkpoint(layer, x, use_reentrant=False)
y.square().sum().backward()
print(y.shape, x.grad.norm().item())

**如何讀結果：**CPU這格檢查可微與輸出；峯值memory取捨要在實際訓練GPU上量測。

**只改一件事：**只用正常layer(x)，覈對相同seed權重的梯度。
